## Ensemble train/test split

In [1]:
import pandas as pd
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report


In [2]:
nltk.download('stopwords')
stop_words = nltk.corpus.stopwords.words('portuguese')

def load_data(file_path):
    return pd.read_json(file_path, lines=True)


def create_x_y(df):
    x = df['text']
    y = df['label']
    return x, y

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
df_train = load_data('corpus/train.jsonl')
x_train, y_train = create_x_y(df_train)

df_val = load_data('corpus/validation.jsonl')
x_val, y_val = create_x_y(df_val)

df_test = load_data('corpus/test.jsonl')
x_test, y_test = create_x_y(df_test)

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words)
x_train_vectorized = vectorizer.fit_transform(x_train)
x_val_vectorized = vectorizer.transform(x_val)
x_test_vectorized = vectorizer.transform(x_test)

In [5]:
rf_model = RandomForestClassifier(n_estimators=100, criterion='entropy', max_depth=15, random_state=40)
lr_model = LogisticRegression(random_state=40, max_iter=100)
svm_model = SVC(probability=True, random_state=40)

voting_model = VotingClassifier(estimators=[
    ('rf', rf_model),
    ('lr', lr_model),
    ('svm', svm_model)
], voting='soft', n_jobs=30)

voting_model.fit(x_train_vectorized, y_train)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(criterion='entropy',
                                                     max_depth=15,
                                                     random_state=40)),
                             ('lr', LogisticRegression(random_state=40)),
                             ('svm', SVC(probability=True, random_state=40))],
                 n_jobs=30, voting='soft')

In [6]:
y_test_pred = voting_model.predict(x_test_vectorized)

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.47      0.70      0.56       570
           1       0.42      0.22      0.29       570

    accuracy                           0.46      1140
   macro avg       0.45      0.46      0.43      1140
weighted avg       0.45      0.46      0.43      1140



## Ensemble cross-validation

In [7]:
from sklearn.model_selection import cross_val_score

In [8]:
data = pd.concat([df_train, df_val, df_test])

X, y = create_x_y(data)

In [9]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words)
x_train_vectorized = vectorizer.fit_transform(X)

In [10]:
scores = cross_val_score(voting_model, x_train_vectorized, y, cv=5)
print(scores.mean())

0.7194736842105262
